# Supervised live Unity lifecycle evidence

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
from contract import ROOT,read_json,sha
run=ROOT/'outputs/runs/checkpoint_check/attempt-01'
result=read_json(run/'result.json')
assert result['status']=='PASS' and len(result['stages'])==3
for stage in result['stages']:
    assert stage['exit_code']==0 and not stage['timed_out'] and not stage['unity_alive_after_worker'] and stage['port_reusable'] and stage['no_tcp_connection_after']
    runtime=read_json(run/stage['stage']/'runtime_result.json')
    assert runtime['unity_exited'] and runtime['static_inputs_preserved'] and not runtime['errors']
print('Verified NEW live training and two fresh evaluations:',json.dumps(result,indent=2))
print('Supervised live Unity lifecycle evidence definitions/execution completed.')


Frozen runtime contract definitions/execution completed.
Verified NEW live training and two fresh evaluations: {
  "status": "PASS",
  "run_id": "checkpoint_check",
  "attempt_id": "attempt-01",
  "stages": [
    {
      "stage": "train",
      "worker_pid": 1700306,
      "command": [
        "/home/obidit/900_PrePreDefense/trace-lab/outputs/environments/runtime-verified/bin/python",
        "-B",
        "-c",
        "from pathlib import Path\nimport importlib.abc, importlib.util, json, sys\nTRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())\nMODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}\nclass NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):\n    def find_spec(self, fullname, path=None, target=None):\n        name = MODULES.get(fullname, fullname)\n        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'\n        if '.' not in fullname 